In [ ]:
!nvidia-smi

Thu Nov 19 04:14:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Google Drive: data, results and logs/ are kept in MyDrive/llmrouter_encoder/ 
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['WORK'] = '/content/drive/MyDrive/llmrouter_encoder'  
os.environ['REPO'] = '/content/llm-routing-study'       
os.makedirs(os.environ['WORK'], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
set -e
if [ -d "$REPO/.git" ]; then
  cd "$REPO" && git fetch -q && git checkout -q encoder && git pull -q
else
  git clone -q -b encoder https://github.com/rahulkanagaraj786/llm-routing-study.git "$REPO"
fi
cd "$REPO" && git log --oneline -1

6442d62 Encoder: progress bar every 20 s with whole-run estimate


In [ ]:
%%bash
cd "$REPO"
pip install -q -r requirements.txt
pip uninstall -y -q torchaudio torchvision torchcodec
if ! python -c "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)" 2>/dev/null; then
  echo "The pinned torch build cannot use this GPU driver; installing torch 2.14.1 for CUDA 12.6"
  pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
fi
python -c "import torch, transformers; from transformers.models.modernbert import modeling_modernbert; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0), '| native bf16:', torch.cuda.is_bf16_supported(including_emulation=False))"

torch 2.15.0+cu132 | transformers 5.23.0 | GPU: Tesla T4 | native bf16: False


In [16]:
%%bash
set -e
cd "$REPO"
mkdir -p data/processed
if [ -f "$WORK/data/processed/paired_v2.jsonl.gz" ]; then
  cp "$WORK"/data/processed/* data/processed/
  echo "data copied from Drive"
else
  mkdir -p data/raw/extracted
  curl -sSL -o data/raw/bench-release.tar.gz "https://huggingface.co/datasets/NPULH/LLMRouterBench/resolve/0e5af1b84bf73437a01a1849c0f1d2468baa93fc/bench-release.tar.gz"
  echo "b79f8cde1a6f029c2efa663a3a3b6f7748defb22341fe59f328cebef6648c8f1  data/raw/bench-release.tar.gz" | sha256sum -c
  tar -xzf data/raw/bench-release.tar.gz -C data/raw/extracted -T data/selected_members.txt
  python scripts/01_build_paired.py | tail -3
  mkdir -p "$WORK/data/processed" && cp data/processed/* "$WORK/data/processed/"
  rm -rf data/raw
  echo "data built and saved to Drive"
fi

data copied from Drive


In [ ]:
%%bash
set -e
cd "$REPO"
for d in results/dev_selection/encoder results/final_dryrun/encoder results/final/encoder logs; do
  mkdir -p "$WORK/$d" "$(dirname "$d")"
  if [ -d "$d" ] && [ ! -L "$d" ]; then cp -rn "$d"/. "$WORK/$d"/ && rm -rf "$d"; fi   # keep anything already written
  [ -L "$d" ] || ln -s "$WORK/$d" "$d"
  echo "$d -> $(readlink "$d")"
done

results/dev_selection/encoder -> /content/drive/MyDrive/llmrouter_encoder/results/dev_selection/encoder
results/final_dryrun/encoder -> /content/drive/MyDrive/llmrouter_encoder/results/final_dryrun/encoder
results/final/encoder -> /content/drive/MyDrive/llmrouter_encoder/results/final/encoder
logs -> /content/drive/MyDrive/llmrouter_encoder/logs


In [ ]:
%%bash
# Tests and smoke check
cd "$REPO"
python tests/test_policy.py && python tests/test_guards.py && python tests/test_models.py
python scripts/05_dev_compare.py encoder --smoke

ok test_select_matches_bruteforce
ok test_endpoints_and_definitions
ok test_per_benchmark_and_bootstrap
ok test_shuffle_and_reweight
ok test_sealed_rows_are_refused
ok test_source_pool_dev_is_allowed_filtered_and_disjoint_from_test
ok test_rq1_dev_covers_all_benchmarks_and_never_the_test_split
  [fine-tune #1] lr 2e-05 | 180 training questions, 60 validation | 12 batches per epoch, up to 3 epochs | cuda, fp16-mixed
    epoch 0.25: train loss 1.0909  val advantage MSE 0.48330  *best  (1s, cuda, fp16-mixed)
    epoch 0.50: train loss 1.0215  val advantage MSE 0.35898  *best  (2s, cuda, fp16-mixed)
    epoch 0.75: train loss 0.9391  val advantage MSE 0.26313  *best  (3s, cuda, fp16-mixed)
    epoch 1.00: train loss 0.8990  val advantage MSE 0.21280  *best  (3s, cuda, fp16-mixed)
    epoch 1.25: train loss 0.4828  val advantage MSE 0.16293  *best  (0s, cuda, fp16-mixed)
    epoch 1.50: train loss 0.4785  val advantage MSE 0.11899  *best  (1s, cuda, fp16-mixed)
    epoch 1.75: train loss 0.

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 4122.97it/s]


In [ ]:
%%bash
# Development run
cd "$REPO"
export TRANSFORMERS_VERBOSITY=error HF_HUB_DISABLE_PROGRESS_BARS=1 TOKENIZERS_PARALLELISM=false
n_done=$(ls results/dev_selection/encoder/rotation_*.json 2>/dev/null | wc -l | tr -d ' ')
echo "Started $(date '+%H:%M:%S')"
echo "Plan: 5 rotations = 6 fine-tunes (rotation 0 tries lr 2e-05 and 5e-05, rotations 1-4 reuse the better one)."
echo "Rotations already finished and saved on Drive: $n_done of 5"
if [ "$n_done" -eq 0 ]; then export ENCODER_TOTAL_FITS=6; else export ENCODER_TOTAL_FITS=$((5 - n_done)); fi
export ENCODER_PROGRESS_SECONDS=20   # a progress line every 20 seconds
echo "Fine-tunes to run now: $ENCODER_TOTAL_FITS"
python -u scripts/05_dev_compare.py encoder 2>&1 \
  | grep --line-buffered -v "unauthenticated requests to the HF Hub" \
  | tee -a "$WORK/dev_encoder.log"
echo "Finished $(date '+%H:%M:%S'). Rotations saved: $(ls results/dev_selection/encoder/rotation_*.json 2>/dev/null | wc -l | tr -d ' ') of 5"


Started 04:16:42
Plan: 5 rotations = 6 fine-tunes (rotation 0 tries lr 2e-05 and 5e-05, rotations 1-4 reuse the better one).
Rotations already finished and saved on Drive: 4 of 5
Fine-tunes to run now: 1
rotation 0: loaded from rotation_0.json
rotation 1: loaded from rotation_1.json
rotation 2: loaded from rotation_2.json
rotation 3: loaded from rotation_3.json
rotation 4: fitting encoder {'lr': 2e-05} on 4973 rows
  [fine-tune #1 of 1] lr 2e-05 | 4,973 training questions, 1,669 validation | 311 batches per epoch, up to 3 epochs | cuda, fp16-mixed
    [#...................]   5% of fine-tune | epoch 1/3 batch 50/311 | 20 s elapsed, at most ~5.9 min left || whole run [#.........]   5%, at most ~5.9 min left
    epoch 0.25: train loss 0.7560  val advantage MSE 0.22000  *best  (30s, cuda, fp16-mixed)
    [##..................]   8% of fine-tune | epoch 1/3 batch 79/311 | 40 s elapsed, at most ~7.3 min left || whole run [#.........]   8%, at most ~7.3 min left
    [###.................]  1

In [ ]:
%%bash
cd "$REPO"
python scripts/07_final_eval.py encoder --plan
python scripts/07_final_eval.py encoder --dry-run 2>&1 | tee -a "$WORK/dryrun_encoder.log"
python scripts/09_report_final.py encoder --dry-run

RQ1 main study (router trained on all seven benchmarks)
  fit         6216  {'aime': 36, 'gpqa': 119, 'hle': 976, 'livecodebench': 633, 'livemathbench': 70, 'mmlupro': 1797, 'simpleqa': 2585}
  cutoffs     2091  {'aime': 11, 'gpqa': 39, 'hle': 336, 'livecodebench': 211, 'livemathbench': 23, 'mmlupro': 597, 'simpleqa': 874}
  test        2060  {'aime': 11, 'gpqa': 40, 'hle': 319, 'livecodebench': 211, 'livemathbench': 24, 'mmlupro': 599, 'simpleqa': 856}
  test, HLE costs repaired  2167  {'aime': 11, 'gpqa': 40, 'hle': 426, 'livecodebench': 211, 'livemathbench': 24, 'mmlupro': 599, 'simpleqa': 856}
model: encoder  study: RQ1  settings: {'lr': 2e-05}  mode: DRY RUN
[rq1] fit on 4980 rows, calibrate on 1662 rows
Loading weights: 100%|██████████| 136/136 [00:00<00:00, 14913.08it/s]
  [fine-tune #1] lr 2e-05 | 4,980 training questions, 1,662 validation | 312 batches per epoch, up to 3 epochs | cuda, fp16-mixed
    [#...................]   5% of fine-tune | epoch 1/3 batch 50/312 | 20 s elap

In [ ]:
import subprocess
subprocess.run('cd "$REPO" && python scripts/07_final_eval.py encoder --unlock-final 2>&1 | tee -a "$WORK/final_encoder.log" '
                   '&& python scripts/09_report_final.py encoder', shell=True, check=True, executable='/bin/bash')